# NB08 – Generator: Würstchen (`wuerstchen`)

**Replaces Stable Cascade**, which cannot run on T4 (requires native bfloat16 conv ops).

**Why Würstchen:**
- Same 42x hierarchical latent compression architecture as Stable Cascade (its predecessor)
- Fully **float16**, runs cleanly on T4 within 15.6 GB VRAM
- No gating, no license acceptance required
- Architecturally distinct from all other generators in the dataset (VQGAN + DiT prior)
- Paper: *Würstchen: An Efficient Architecture for Large-Scale Text-to-Image Diffusion Models* (ICLR 2024)

**What changes in the dataset:**
- Generator key: `sd_cascade` → `wuerstchen`
- All existing corrupted sd_cascade shards (5KB black images) are deleted first
- New images stored at `data/wuerstchen/`
- `config.json` updated to replace `sd_cascade` with `wuerstchen`

### Prerequisites
NB00, NB01, NB02 finished. Internet **ON**, GPU **ON** (T4). `HF_TOKEN` set.

In [1]:
import importlib.util, sys, subprocess, os, shutil

CACHE_ROOT = '/kaggle/temp/hf_cache'
os.environ['HF_HOME']      = CACHE_ROOT
os.environ['HF_HUB_CACHE'] = CACHE_ROOT + '/hub'
if importlib.util.find_spec('hf_transfer'):
    os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
else:
    os.environ.pop('HF_HUB_ENABLE_HF_TRANSFER', None)
os.makedirs(CACHE_ROOT, exist_ok=True)

for mnt, lbl in [('/kaggle/temp','scratch'), ('/kaggle/working','working')]:
    try:
        t, u, f = shutil.disk_usage(mnt)
        print(f'{lbl:12s} free={f/2**30:6.1f} GB / total={t/2**30:6.1f} GB')
    except FileNotFoundError:
        print(f'{lbl}: not present')

import PIL._typing
if not hasattr(PIL._typing, '_Ink'):
    from typing import Union
    PIL._typing._Ink = Union[int, float, str, bytes, tuple]
    for _k in list(sys.modules):
        if _k in ('PIL.ImageText','PIL.ImageDraw') or _k.startswith('torchvision'):
            del sys.modules[_k]
    print('patched PIL._typing._Ink')

def _pip(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)

need = []
try:
    import diffusers as _d
    if tuple(int(x) for x in _d.__version__.split('.')[:2]) < (0, 30):
        need.append('diffusers>=0.30')
except ImportError:
    need.append('diffusers>=0.30')
for m, p in [('safetensors','safetensors'),('sentencepiece','sentencepiece'),
             ('protobuf','protobuf')]:
    if not importlib.util.find_spec(m): need.append(p)
if need: _pip(*need); print('installed:', need)
else: print('all deps present')

import diffusers, torch
print(f'diffusers {diffusers.__version__} | torch {torch.__version__} | cuda: {torch.cuda.is_available()}')

scratch      free=1102.1 GB / total=8062.4 GB
working      free=  19.5 GB / total=  19.5 GB
patched PIL._typing._Ink
installed: ['protobuf']
diffusers 0.37.1 | torch 2.10.0+cu128 | cuda: True


In [2]:
REPO_ID    = 'Shanmuk4622/ai-detection-dataset-v2'
MODEL      = 'wuerstchen'           # new generator key
OLD_MODEL  = 'sd_cascade'           # corrupted shards to delete

import os
from huggingface_hub import hf_hub_download

def load_token():
    try:
        from kaggle_secrets import UserSecretsClient
        t = UserSecretsClient().get_secret('HF_TOKEN')
        if t: return t.strip()
    except Exception: pass
    for k in ('HF_TOKEN','HUGGINGFACE_TOKEN','HUGGING_FACE_HUB_TOKEN'):
        if os.environ.get(k): return os.environ[k].strip()
    import getpass; return getpass.getpass('HF write token: ').strip()

TOKEN = load_token()
os.environ['HF_TOKEN'] = TOKEN

lib  = hf_hub_download(REPO_ID, 'ai_detect_common.py', repo_type='dataset', token=TOKEN)
import importlib.util
spec = importlib.util.spec_from_file_location('ai_detect_common', lib)
C    = importlib.util.module_from_spec(spec); spec.loader.exec_module(C)
cfg  = C.read_config(REPO_ID, TOKEN)
MASTER_SEED = cfg.get('master_seed', 42)
print(f'library {C.PIPELINE_VERSION} | MASTER_SEED={MASTER_SEED}')
print('generators in config:', list(cfg['generators'].keys()))

ai_detect_common.py: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

library 1.2 | MASTER_SEED=42
generators in config: ['sd15', 'sdxl', 'flux_schnell', 'kandinsky22', 'pixart_sigma', 'wuerstchen']


## Step 1 — Delete corrupted sd_cascade shards + update config

All existing `data/sd_cascade/` shards contain black images (5KB). Delete them.
Then patch `config.json` to replace `sd_cascade` with `wuerstchen`.

In [3]:
import json
from huggingface_hub import list_repo_files, CommitOperationDelete, CommitOperationAdd

api = C.HfApi(token=TOKEN)

# ── 1a. Delete corrupted sd_cascade shards ────────────────────────────────────
all_files = list(list_repo_files(REPO_ID, repo_type='dataset', token=TOKEN))
bad_shards = [f for f in all_files
              if f.startswith(f'data/{OLD_MODEL}/') and f.endswith('.parquet')]
print(f'Found {len(bad_shards)} corrupted {OLD_MODEL} shards to delete')

ops = [CommitOperationDelete(path_in_repo=f) for f in bad_shards]

# ── 1b. Update config.json: replace sd_cascade entry with wuerstchen ──────────
import io, tempfile
cfg_local = hf_hub_download(REPO_ID, 'config.json', repo_type='dataset', token=TOKEN)
cfg_data  = json.loads(open(cfg_local).read())

# Remove sd_cascade, add wuerstchen
cascade_spec = cfg_data['generators'].pop('sd_cascade', {})
cfg_data['generators']['wuerstchen'] = {
    'model_id':   'warp-ai/wuerstchen',
    'prior_id':   'warp-ai/wuerstchen-prior',
    'pipeline':   'wuerstchen',
    'native':     1024,
    'steps':      [20, 10],       # prior steps, decoder steps
    'guidance':   [4.0, 0.0],     # prior guidance, decoder guidance
}
print('Updated config generators:', list(cfg_data['generators'].keys()))

new_cfg_bytes = json.dumps(cfg_data, indent=2).encode()
ops.append(CommitOperationAdd(
    path_in_repo='config.json',
    path_or_fileobj=io.BytesIO(new_cfg_bytes)
))

# ── Commit all in one shot ─────────────────────────────────────────────────────
if ops:
    # Delete in batches of 49 (leave 1 slot for the config add)
    delete_ops = [o for o in ops if isinstance(o, CommitOperationDelete)]
    add_ops    = [o for o in ops if isinstance(o, CommitOperationAdd)]

    for i in range(0, len(delete_ops), 49):
        batch = delete_ops[i:i+49] + (add_ops if i == 0 else [])
        api.create_commit(
            repo_id=REPO_ID, repo_type='dataset', operations=batch,
            commit_message=f'NB08: delete sd_cascade shards + add wuerstchen config (batch {i//49+1})'
        )
        print(f'  Committed batch {i//49+1}: {len(batch)} ops')

    if not delete_ops:  # only config to add
        api.create_commit(
            repo_id=REPO_ID, repo_type='dataset', operations=add_ops,
            commit_message='NB08: add wuerstchen to config'
        )

# Reload config
cfg = C.read_config(REPO_ID, TOKEN)
g   = cfg['generators']['wuerstchen']
print('\nNew wuerstchen spec:', g)
print('Config updated and committed.')

Found 0 corrupted sd_cascade shards to delete
Updated config generators: ['sd15', 'sdxl', 'flux_schnell', 'kandinsky22', 'pixart_sigma', 'wuerstchen']


No files have been modified since last commit. Skipping to prevent empty commit.



New wuerstchen spec: {'model_id': 'warp-ai/wuerstchen', 'prior_id': 'warp-ai/wuerstchen-prior', 'pipeline': 'wuerstchen', 'native': 1024, 'steps': [20, 10], 'guidance': [4.0, 0.0]}
Config updated and committed.


## Step 2 — Load Würstchen pipeline

Single `AutoPipelineForText2Image` call — no manual prior/decoder separation needed.
Pure float16, no bfloat16 required. Both prior and decoder are loaded together.

In [4]:
import torch
from diffusers import WuerstchenPriorPipeline, WuerstchenDecoderPipeline
from diffusers.pipelines.wuerstchen import DEFAULT_STAGE_C_TIMESTEPS

DTYPE  = torch.float16
native = g['native']      # 1024
p_guid = g['guidance'][0] # 4.0 (prior)
d_guid = g['guidance'][1] # 0.0 (decoder)

print('Loading WuerstchenPriorPipeline (float16)...')
prior_pipe = WuerstchenPriorPipeline.from_pretrained(
    g['prior_id'],
    torch_dtype=DTYPE,
).to('cuda')
prior_pipe.set_progress_bar_config(disable=True)
print('Prior ready.')

print('Loading WuerstchenDecoderPipeline (float16)...')
decoder_pipe = WuerstchenDecoderPipeline.from_pretrained(
    g['model_id'],
    torch_dtype=DTYPE,
).to('cuda')
decoder_pipe.set_progress_bar_config(disable=True)
print('Decoder ready.')

vram_used = torch.cuda.memory_allocated() / 2**30
vram_total = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f'VRAM used: {vram_used:.1f} / {vram_total:.1f} GB')

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Loading WuerstchenPriorPipeline (float16)...


model_index.json:   0%|          | 0.00/420 [00:00<?, ?B/s]

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/4 [00:00<?, ?it/s]

...

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/388 [00:00<?, ?it/s]

The WuerstchenDecoderPipeline has been deprecated and will not receive bug fixes or feature updates after Diffusers version None. 


Decoder ready.
VRAM used: 5.8 / 14.6 GB


In [5]:
import numpy as np
from PIL import Image

def is_valid_image(pil_img):
    arr  = np.asarray(pil_img).astype(np.float32)
    std  = float(arr.std())
    mean = float(arr.mean())
    if std  < 8.0:   return False, f'std={std:.1f} (uniform/black)'
    if mean < 2.0:   return False, f'mean={mean:.1f} (all black)'
    if mean > 253.0: return False, f'mean={mean:.1f} (all white)'
    return True, 'ok'

def generate(prompt, seed):
    gn = torch.Generator(device='cpu').manual_seed(seed)

    # Stage 1: Prior
    prior_output = prior_pipe(
        prompt=prompt,
        height=native, width=native,
        timesteps=DEFAULT_STAGE_C_TIMESTEPS,
        negative_prompt='',
        guidance_scale=p_guid,
        num_images_per_prompt=1,
        generator=gn,
    )

    # Stage 2: Decoder
    decoder_output = decoder_pipe(
        image_embeddings=prior_output.image_embeddings,
        prompt=prompt,
        negative_prompt='',
        guidance_scale=d_guid,
        output_type='pil',
        generator=gn,
    )
    return decoder_output.images[0]

# Smoke test
print('Smoke test: generating 1 image...')
_test = generate('a wooden dock next to a lake at sunset', seed=99999)
_ok, _why = is_valid_image(_test)
print(f'  size={_test.size}  valid={_ok}  reason={_why}')
if not _ok:
    raise RuntimeError(f'Smoke test FAILED: {_why}')
print('Smoke test PASSED. Proceeding to generation.')

Smoke test: generating 1 image...
  size=(1024, 1024)  valid=True  reason=ok
Smoke test PASSED. Proceeding to generation.


## Step 3 — Generate 10,000 images

In [ ]:
import gc, time, hashlib

TARGET = cfg['target_per_generator']   # 10_000

captions = {}
for f in C.list_shards(REPO_ID, 'captions/', TOKEN):
    local = C.hf_hub_download(REPO_ID, f, repo_type='dataset', token=TOKEN)
    t = C.pq.read_table(local, columns=['source_real_id', 'caption'])
    for sid, cap in zip(t.column('source_real_id').to_pylist(),
                        t.column('caption').to_pylist()):
        captions[sid] = cap
assert captions, 'No captions found — run NB02 first.'
print(f'Captions: {len(captions)}')

NUM_WORKERS = 5
WORKER_ID   = 0   # <<< CHANGE PER ACCOUNT: 0, 1, or 2
assert 0 <= WORKER_ID < NUM_WORKERS

def _owner(sid):
    return int(hashlib.sha256(str(sid).encode()).hexdigest(), 16) % NUM_WORKERS

universe = sorted(captions)[:TARGET]
mine     = [sid for sid in universe if _owner(sid) == WORKER_ID]
done     = C.reconcile_ids(REPO_ID, f'data/{MODEL}/', TOKEN)
todo     = [sid for sid in mine if sid not in done]
print(f'worker {WORKER_ID}/{NUM_WORKERS} | my={len(mine)} | '
      f'done(global)={len(done)} | remaining={len(todo)} | target={TARGET}')

writer = C.ShardWriter(api, REPO_ID, f'data/{MODEL}/',
                       commit_interval=cfg['commit_interval_seconds'],
                       max_rows=cfg['batch_size'], token=TOKEN)

accepted, failed, skipped = 0, 0, 0
t0 = time.time()

try:
    for sid in todo:
        prompt = captions[sid]
        seed   = C.make_seed(MODEL, sid, MASTER_SEED)

        try:
            img = generate(prompt, seed)
        except Exception as e:
            failed += 1
            if failed <= 10 or failed % 50 == 0:
                print(f'  [FAIL] {sid}: {type(e).__name__}: {e}')
            continue

        ok, reason = is_valid_image(img)
        if not ok:
            skipped += 1
            if skipped <= 10 or skipped % 50 == 0:
                print(f'  [SKIP] {sid}: {reason}')
            continue

        png = C.canonical_preprocess(img)
        num = str(sid).split('_')[-1]
        row = C.empty_row()
        row.update(dict(
            image_id         = f'{MODEL}_{num}',
            source_real_id   = sid,
            label            = 1,
            generator        = MODEL,
            source_dataset   = MODEL,
            prompt           = prompt,
            image            = png,
            width            = C.TARGET,
            height           = C.TARGET,
            orig_width       = native,
            orig_height      = native,
            gen_model_id     = g['model_id'],
            gen_steps        = int(g['steps'][0]),
            gen_guidance     = float(g['guidance'][0]),
            gen_native_res   = int(native),
            seed             = int(seed),
            caption_model    = cfg['caption_model'],
            pipeline_version = C.PIPELINE_VERSION,
            sha256           = C.sha256_bytes(png),
            created_utc      = C.now_utc()
        ))
        writer.add(row)
        accepted += 1

        if accepted % 50 == 0:
            elapsed = time.time() - t0
            rate    = accepted / elapsed
            eta_h   = ((len(mine) - accepted) / rate / 3600) if rate > 0 else 0
            print(f'  w{WORKER_ID}: {accepted}/{len(mine)} | '
                  f'{rate:.2f} img/s | ETA {eta_h:.1f}h | '
                  f'failed={failed} skipped={skipped}')
            gc.collect()

        writer.maybe_flush()

finally:
    writer.close()

total = len(done) + accepted
print(f'Done. accepted={accepted} failed={failed} skipped={skipped} total={total}/{TARGET}')

captions/captions-906881cf-00000.parquet:   0%|          | 0.00/23.2k [00:00<?, ?B/s]

captions/captions-906881cf-00001.parquet:   0%|          | 0.00/23.1k [00:00<?, ?B/s]

captions/captions-906881cf-00002.parquet:   0%|          | 0.00/23.3k [00:00<?, ?B/s]

captions/captions-906881cf-00003.parquet:   0%|          | 0.00/23.5k [00:00<?, ?B/s]

captions/captions-906881cf-00004.parquet:   0%|          | 0.00/23.1k [00:00<?, ?B/s]

...

New Data Upload: |          |  0.00B /  0.00B            

  committed 106 rows (848 total) -> data/wuerstchen/wuerstchen-bc6bb59b-00007.parquet
  w0: 850/2030 | 0.09 img/s | ETA 3.8h | failed=0 skipped=0
  w0: 900/2030 | 0.09 img/s | ETA 3.6h | failed=0 skipped=0
  w0: 950/2030 | 0.09 img/s | ETA 3.4h | failed=0 skipped=0


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

  committed 106 rows (954 total) -> data/wuerstchen/wuerstchen-bc6bb59b-00008.parquet
  w0: 1000/2030 | 0.09 img/s | ETA 3.3h | failed=0 skipped=0


## Self-check

In [ ]:
import random, numpy as np, io
from PIL import Image as _PIL

n = C.count_rows(REPO_ID, f'data/{MODEL}/', TOKEN)
print(f'{MODEL}: {n} rows (target {cfg["target_per_generator"]})')

shards = C.list_shards(REPO_ID, f'data/{MODEL}/', TOKEN)
if shards:
    loc = C.hf_hub_download(REPO_ID, random.choice(shards),
                             repo_type='dataset', token=TOKEN)
    t   = C.pq.read_table(loc)
    j   = random.randrange(t.num_rows)
    png = t.column('image')[j].as_py()

    ok_c, why = C.png_is_canonical(png)
    img  = _PIL.open(io.BytesIO(png)).convert('RGB')
    arr  = np.asarray(img).astype(np.float32)
    size_kb = len(png) // 1024

    print(f'Sample : {t.column("image_id")[j].as_py()}')
    print(f'  canonical : {ok_c} ({why})')
    print(f'  PNG size  : {size_kb} KB  (must be >50 KB for real images)')
    print(f'  pixel std : {arr.std():.1f}  (must be >8)')
    print(f'  pixel mean: {arr.mean():.1f}')

    if size_kb < 50 or arr.std() < 8:
        print('FAIL: generation is still producing bad images')
    else:
        print('PASS: images look correct')
else:
    print('No shards yet.')